# SMPL Mode 1 Workflow Demo

This notebook provisions an SMPL Mode 1 runtime, runs fixed-topology differentiation on pose/shape parameters, writes artifacts to disk, and shows the resulting numbers.

By default it uses a small in-notebook toy model so it runs without external SMPL assets. You can switch to a real model path by changing the setup cell.


In [ ]:
from pathlib import Path
import json
import sys

import jax.numpy as jnp

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from smpljax import initialize_mode1_model, run_mode1_workflow, visualize_mode1_result
from smpljax.body_models import SMPLJAXModel
from smpljax.utils import SMPLModelData

OUTPUT_ROOT = REPO_ROOT / "outputs/smpl/notebooks/mode1_workflow_demo"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

USE_TOY_MODEL = True
MODEL_PATH = None  # set this to a real .npz or .pkl model file when USE_TOY_MODEL = False
VISUALIZE = False
VISUALIZE_BACKEND = "matplotlib"

print(f"repo root: {REPO_ROOT}")
print(f"output root: {OUTPUT_ROOT}")


In [ ]:
def make_toy_model() -> SMPLJAXModel:
    return SMPLJAXModel(
        data=SMPLModelData(
            v_template=jnp.array(
                [
                    [0.0, 0.0, 0.0],
                    [1.0, 0.0, 0.0],
                    [1.0, 1.0, 0.0],
                    [0.0, 1.0, 0.0],
                ],
                dtype=jnp.float32,
            ),
            shapedirs=jnp.zeros((4, 3, 10), dtype=jnp.float32),
            posedirs=jnp.zeros((27, 12), dtype=jnp.float32),
            j_regressor=jnp.ones((4, 4), dtype=jnp.float32) / 4.0,
            parents=jnp.array([-1, 0, 1, 2], dtype=jnp.int32),
            lbs_weights=jnp.ones((4, 4), dtype=jnp.float32) / 4.0,
            num_betas=10,
            num_body_joints=3,
            num_hand_joints=0,
            num_face_joints=0,
            faces_tensor=jnp.array([[0, 1, 2], [0, 2, 3]], dtype=jnp.int32),
        )
    )


if USE_TOY_MODEL:
    model_source = {"model": make_toy_model()}
else:
    if MODEL_PATH is None:
        raise ValueError("Set MODEL_PATH when USE_TOY_MODEL is False")
    model_source = {"model_path": MODEL_PATH, "runtime_mode": "optimized"}

model_source


## 1. Provision The SMPL Mode 1 Runtime

This creates the runtime and default differentiable parameter tree.


In [ ]:
provision = initialize_mode1_model(
    batch_size=1,
    progress=True,
    **model_source,
)
print(type(provision.model).__name__)
print(sorted(provision.params.keys()))
{key: tuple(value.shape) for key, value in provision.params.items()}


## 2. Set A Non-Zero Initial Translation

The default objective in SMPL Mode 1 penalizes vertex magnitude and translation magnitude, so a non-zero initial translation gives a visible optimization trajectory.


In [ ]:
params = dict(provision.params)
params["transl"] = jnp.array([[0.5, -0.25, 0.1]], dtype=jnp.float32)
params["transl"]


## 3. Run SMPL Mode 1

This differentiates through the SMPL forward pass, optimizes the active parameters, and writes artifacts to `outputs/...`.


In [ ]:
run = run_mode1_workflow(
    provision,
    output_dir=OUTPUT_ROOT,
    prefix="smpl_mode1_demo",
    params=params,
    steps=12,
    step_size=0.2,
    diagnostics_every=3,
    progress=True,
)
run.artifacts


## 4. Show Numbers

Read the written metric and history files to inspect the optimization behavior.


In [ ]:
metrics = json.loads(run.artifacts["metrics"].read_text(encoding="utf-8"))
history = json.loads(run.artifacts["history_json"].read_text(encoding="utf-8"))

summary = {
    "runtime_mode": provision.runtime_mode,
    "n_steps": metrics["n_steps"],
    "initial_objective": metrics["initial_objective"],
    "final_objective": metrics["final_objective"],
    "initial_grad_norm": metrics["initial_grad_norm"],
    "final_grad_norm": metrics["final_grad_norm"],
    "vertex_rms": metrics["vertex_rms"],
    "joint_rms": metrics["joint_rms"],
    "history_rows": len(history),
}
summary


## 5. Write An Artifact Summary File


In [ ]:
artifact_summary = {
    "metrics": str(run.artifacts["metrics"]),
    "history_json": str(run.artifacts["history_json"]),
    "history_csv": str(run.artifacts["history_csv"]),
    "snapshot": str(run.artifacts["snapshot"]),
    "mode1_snapshot": str(run.artifacts["mode1_snapshot"]),
    "viewer_payload": str(run.artifacts["viewer_payload"]),
}

artifact_summary_path = OUTPUT_ROOT / "artifact_summary.json"
artifact_summary_path.write_text(json.dumps(artifact_summary, indent=2), encoding="utf-8")
artifact_summary


## 6. Optional Visualization

Set `VISUALIZE = True` to dispatch to the selected backend.


In [ ]:
if VISUALIZE:
    viewer = visualize_mode1_result(run.result, backend=VISUALIZE_BACKEND)
    viewer
else:
    print("Set VISUALIZE = True in the setup cell to render the result.")
